# Day2+ — Cross-Board Transfer Stress Test

This notebook extends Day2 with a systematic cross-board transfer stress test for methane (`GMe`) in the UCI #361 Twin Gas Sensor Arrays dataset.

This is **not** Day3 few-shot adaptation. The goal is not benchmark maximization. The goal is to understand transfer structure, transfer asymmetry, source-domain diversity, and whether a drifted/dynamically shifted board can improve robustness when used as a source domain.

Scientific framing: **B5 is not treated as a bad sensor**. It is treated as a drifted, dynamically shifted, broadened-domain, industrially realistic board.

In [4]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Robust repo-root detection: this notebook is expected in MOx_Calibration_Transfer/notebooks/
NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    # Works when executing from project root or from an external runner.
    PROJECT_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "data").exists() else Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data" / "raw"
SRC_DIR = PROJECT_ROOT / "src"
FIG_DIR = PROJECT_ROOT / "figures" / "day2plus"
RESULT_DIR = PROJECT_ROOT / "results" / "day2plus"

for p in [SRC_DIR, FIG_DIR, RESULT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("DATA_DIR:", DATA_DIR.resolve())
print("FIG_DIR:", FIG_DIR.resolve())
print("RESULT_DIR:", RESULT_DIR.resolve())

assert SRC_DIR.exists(), f"Missing src directory: {SRC_DIR}"
assert DATA_DIR.exists(), f"Missing raw data directory: {DATA_DIR}"

PROJECT_ROOT: C:\Users\hg\PycharmProjects\mox_calibration_transfer
DATA_DIR: C:\Users\hg\PycharmProjects\mox_calibration_transfer\data\raw
FIG_DIR: C:\Users\hg\PycharmProjects\mox_calibration_transfer\figures\day2plus
RESULT_DIR: C:\Users\hg\PycharmProjects\mox_calibration_transfer\results\day2plus


In [5]:
from day2plus_transfer_matrix import (
    HAS_XGBOOST,
    TARGET_GAS,
    assert_no_label_leakage,
    build_feature_table,
    compute_asymmetry,
    evaluate_split,
    make_transfer_matrix,
    plot_asymmetry,
    plot_concentration_failure,
    plot_heatmap,
    plot_pca_by_board_role,
    plot_source_diversity,
    run_leave_one_out,
    run_mixed_domain_combinations,
    run_single_source_matrix,
    save_json,
)

warnings.filterwarnings("ignore", category=UserWarning)

MODEL_NAMES = ["LinearRegression", "RandomForest"] + (["XGBoost"] if HAS_XGBOOST else [])
FEATURE_SETS = ["raw", "physics", "combined"]
PRIMARY_MODEL = "RandomForest"
PRIMARY_FEATURE_SET = "physics"

print("Models:", MODEL_NAMES)
print("Feature sets:", FEATURE_SETS)

Models: ['LinearRegression', 'RandomForest', 'XGBoost']
Feature sets: ['raw', 'physics', 'combined']


## 1. Build methane feature table

Feature construction follows the Day2 logic: raw statistics, physics-informed response/recovery descriptors, and a combined representation. Concentration labels are kept only as targets and metadata, never as input features.

In [6]:
df = build_feature_table(DATA_DIR, gas=TARGET_GAS)
print(df.shape)
display(df[["filename", "board", "gas", "concentration_code", "concentration_numeric", "replicate"]].head())
print("Boards:", sorted(df["board"].unique()))
print("Concentrations:", sorted(df["concentration_numeric"].unique()))
print(df.groupby("board").size())

feature_report = assert_no_label_leakage(df)
save_json(feature_report, RESULT_DIR / "feature_leakage_check.json")
feature_report

(160, 126)


,filename,board,gas,concentration_code,concentration_numeric,replicate
0,B1_GMe_F010_R1.txt,B1,GMe,F010,10,R1
1,B1_GMe_F010_R2.txt,B1,GMe,F010,10,R2
2,B1_GMe_F010_R3.txt,B1,GMe,F010,10,R3
3,B1_GMe_F010_R4.txt,B1,GMe,F010,10,R4
4,B1_GMe_F020_R1.txt,B1,GMe,F020,20,R1


Boards: ['B1', 'B2', 'B3', 'B4', 'B5']
Concentrations: [np.int64(10), np.int64(20), np.int64(30), np.int64(40), np.int64(50), np.int64(60), np.int64(70), np.int64(80), np.int64(90), np.int64(100)]
board
B1    40
B2    40
B3    40
B4    20
B5    20
dtype: int64


{'raw': {'n_features': 48, 'leakage_overlap': [], 'ok': True},
 'physics': {'n_features': 71, 'leakage_overlap': [], 'ok': True},
 'combined': {'n_features': 119, 'leakage_overlap': [], 'ok': True}}

**Interpretation checkpoint.** This section verifies the experimental population before modeling. The leakage check is intentionally explicit because cross-board transfer is only meaningful if concentration labels and board metadata are not used as input features.

## 2. Single-source board-to-board transfer matrix

This experiment trains on one board and tests on another board. It directly addresses transfer asymmetry, e.g. `B1 → B5` versus `B5 → B1`.

In [7]:
single_metrics, single_preds = run_single_source_matrix(
    df,
    feature_sets=FEATURE_SETS,
    model_names=MODEL_NAMES,
)

single_metrics.to_csv(RESULT_DIR / "single_source_transfer_metrics.csv", index=False)
single_preds.to_csv(RESULT_DIR / "single_source_transfer_predictions.csv", index=False)

display(single_metrics.sort_values("rmse").head(10))
display(single_metrics.sort_values("rmse", ascending=False).head(10))

,source_boards,n_source_boards,target_board,feature_set,model,n_train,n_test,rmse,mae,r2,includes_B5_source
150,B3,1,B4,combined,RandomForest,40,20,3.939436,3.268750,0.981189,False
90,B3,1,B4,physics,RandomForest,40,20,4.046920,3.245000,0.980148,False
91,B3,1,B5,physics,RandomForest,40,20,4.202220,2.978750,0.978596,False
151,B3,1,B5,combined,RandomForest,40,20,4.288695,2.881250,0.977706,False
147,B2,1,B5,combined,RandomForest,40,20,4.751881,3.907500,0.972630,False
42,B1,1,B4,raw,XGBoost,40,20,4.798241,3.369572,0.972093,False
167,B2,1,B5,combined,XGBoost,40,20,4.834105,3.160089,0.971674,False
162,B1,1,B4,combined,XGBoost,40,20,4.873770,3.549797,0.971208,False
107,B2,1,B5,physics,XGBoost,40,20,4.874446,3.423608,0.971200,False
146,B2,1,B4,combined,RandomForest,40,20,4.976868,4.398750,0.969977,False


,source_boards,n_source_boards,target_board,feature_set,model,n_train,n_test,rmse,mae,r2,includes_B5_source
15,B4,1,B5,raw,LinearRegression,20,20,1336.120393,1261.345500,-2162.900247,False
6,B2,1,B4,raw,LinearRegression,40,20,953.104163,952.844818,-1100.100056,False
0,B1,1,B2,raw,LinearRegression,40,40,726.664588,721.114669,-639.050210,False
135,B4,1,B5,combined,LinearRegression,20,20,651.711238,641.024551,-513.821258,False
123,B1,1,B5,combined,LinearRegression,40,20,648.181401,286.325452,-508.259549,False
4,B2,1,B1,raw,LinearRegression,40,40,643.922178,637.440919,-501.588813,False
12,B4,1,B1,raw,LinearRegression,20,40,503.459122,483.269516,-306.237682,False
75,B4,1,B5,physics,LinearRegression,20,20,479.536294,361.444713,-277.733403,False
63,B1,1,B5,physics,LinearRegression,40,20,464.454555,251.401503,-260.476404,False
2,B1,1,B4,raw,LinearRegression,40,20,453.452603,436.417147,-248.235471,False


In [8]:
for metric, cmap in [("rmse", "magma"), ("mae", "magma"), ("r2", "viridis")]:
    matrix = make_transfer_matrix(single_metrics, metric=metric, feature_set=PRIMARY_FEATURE_SET, model=PRIMARY_MODEL)
    matrix.to_csv(RESULT_DIR / f"single_source_{PRIMARY_MODEL}_{PRIMARY_FEATURE_SET}_{metric}_matrix.csv")
    plot_heatmap(
        matrix,
        title=f"Single-source transfer {metric.upper()} — {PRIMARY_MODEL}, {PRIMARY_FEATURE_SET}",
        path=FIG_DIR / f"single_source_{PRIMARY_MODEL}_{PRIMARY_FEATURE_SET}_{metric}_heatmap.png",
        cmap=cmap,
    )
    display(matrix)

target_board,B1,B2,B3,B4,B5
source_boards,,,,,
B1,NaN,6.237779,10.385157,9.155391,7.793647
B2,8.160622,NaN,7.389068,5.822086,5.068000
B3,12.264505,7.762181,NaN,4.046920,4.202220
B4,17.059493,11.204104,8.915445,NaN,9.522624
B5,12.973678,7.175764,6.514683,5.113347,NaN


target_board,B1,B2,B3,B4,B5
source_boards,,,,,
B1,NaN,4.825625,9.590625,8.2025,7.28750
B2,6.738750,NaN,6.695625,5.0400,4.02250
B3,8.894375,6.038125,NaN,3.2450,2.97875
B4,12.948750,8.363750,7.308750,NaN,7.40250
B5,9.587500,5.845000,5.191250,3.9350,NaN


target_board,B1,B2,B3,B4,B5
source_boards,,,,,
B1,NaN,0.952836,0.869271,0.898399,0.926375
B2,0.919278,NaN,0.933820,0.958913,0.968867
B3,0.817675,0.926968,NaN,0.980148,0.978596
B4,0.647241,0.847840,0.903654,NaN,0.890084
B5,0.795980,0.937586,0.948556,0.968307,NaN


**Interpretation checkpoint.** Read each heatmap row as the source board and each column as the target board. Strong row effects suggest useful source domains; strong column effects suggest intrinsically easy or hard target domains.

## 3. Transfer asymmetry

Transfer is symmetric only if `source → target` performs similarly to `target → source`. In real MOx systems, asymmetry is expected because baseline shift, response amplitude, recovery dynamics, and saturation can deform the domain geometry differently in each direction.

In [9]:
asym = compute_asymmetry(single_metrics, metric="rmse")
asym.to_csv(RESULT_DIR / "single_source_rmse_asymmetry.csv", index=False)
plot_asymmetry(
    asym,
    feature_set=PRIMARY_FEATURE_SET,
    model=PRIMARY_MODEL,
    path=FIG_DIR / f"asymmetry_{PRIMARY_MODEL}_{PRIMARY_FEATURE_SET}.png",
    metric="rmse",
)
display(asym[(asym["feature_set"] == PRIMARY_FEATURE_SET) & (asym["model"] == PRIMARY_MODEL)].sort_values("rmse_abs_asymmetry", ascending=False))

,board_pair,source_to_target,reverse,feature_set,model,rmse_forward,rmse_reverse,rmse_abs_asymmetry,rmse_signed_asymmetry
42,B1<->B4,B1->B4,B4->B1,physics,RandomForest,9.155391,17.059493,7.904102,-7.904102
45,B2<->B4,B2->B4,B4->B2,physics,RandomForest,5.822086,11.204104,5.382017,-5.382017
43,B1<->B5,B1->B5,B5->B1,physics,RandomForest,7.793647,12.973678,5.180030,-5.180030
47,B3<->B4,B3->B4,B4->B3,physics,RandomForest,4.046920,8.915445,4.868525,-4.868525
49,B4<->B5,B4->B5,B5->B4,physics,RandomForest,9.522624,5.113347,4.409278,4.409278
48,B3<->B5,B3->B5,B5->B3,physics,RandomForest,4.202220,6.514683,2.312463,-2.312463
46,B2<->B5,B2->B5,B5->B2,physics,RandomForest,5.068000,7.175764,2.107764,-2.107764
40,B1<->B2,B1->B2,B2->B1,physics,RandomForest,6.237779,8.160622,1.922843,-1.922843
41,B1<->B3,B1->B3,B3->B1,physics,RandomForest,10.385157,12.264505,1.879348,-1.879348
44,B2<->B3,B2->B3,B3->B2,physics,RandomForest,7.389068,7.762181,0.373113,-0.373113


**Interpretation checkpoint.** Large asymmetry means the two boards do not merely differ by a reversible offset. It suggests board-specific deformation of the calibration manifold.

## 4. Leave-one-board-out mixed-domain training

This is the Day2-style regime generalized to all boards: train on all boards except one, then test on the held-out board. This addresses which boards are hardest to transfer **to** when the source population is broad.

In [10]:
loo_metrics, loo_preds = run_leave_one_out(
    df,
    feature_sets=FEATURE_SETS,
    model_names=MODEL_NAMES,
)
loo_metrics.to_csv(RESULT_DIR / "leave_one_board_out_metrics.csv", index=False)
loo_preds.to_csv(RESULT_DIR / "leave_one_board_out_predictions.csv", index=False)

display(loo_metrics.sort_values(["feature_set", "model", "rmse"]))

,source_boards,n_source_boards,target_board,feature_set,model,n_train,n_test,rmse,mae,r2,includes_B5_source
31,B1+B3+B4+B5,4,B2,combined,LinearRegression,120,40,37.583498,29.872817,-0.712145,True
33,B1+B2+B3+B5,4,B4,combined,LinearRegression,140,20,86.858393,78.751056,-8.144704,True
32,B1+B2+B4+B5,4,B3,combined,LinearRegression,120,40,110.724051,106.804202,-13.860382,True
30,B2+B3+B4+B5,4,B1,combined,LinearRegression,120,40,114.123579,113.023805,-14.786899,True
34,B1+B2+B3+B4,4,B5,combined,LinearRegression,140,20,428.763373,375.959097,-221.833975,False
38,B1+B2+B3+B5,4,B4,combined,RandomForest,140,20,4.228013,3.456250,0.978332,True
39,B1+B2+B3+B4,4,B5,combined,RandomForest,140,20,4.618005,3.096250,0.974150,False
37,B1+B2+B4+B5,4,B3,combined,RandomForest,120,40,5.024069,4.190625,0.969405,True
36,B1+B3+B4+B5,4,B2,combined,RandomForest,120,40,5.534648,4.313125,0.962870,True
35,B2+B3+B4+B5,4,B1,combined,RandomForest,120,40,9.264736,6.778125,0.895957,True


**Interpretation checkpoint.** If B5 remains difficult as a target even when trained on B1+B2+B3+B4, then B5 is not just shifted; it occupies a harder target geometry. If another board becomes easier when B5 is included as source, B5 may still be useful for robustness.

## 5. Source-domain diversity and drift-inclusive robustness

This section trains on all 2-, 3-, and 4-board source combinations and tests on the remaining target board. It compares clean-only source sets versus B5-inclusive source sets.

In [11]:
combo_metrics, combo_preds = run_mixed_domain_combinations(
    df,
    source_sizes=[2, 3, 4],
    feature_set=PRIMARY_FEATURE_SET,
    model_name=PRIMARY_MODEL,
)
combo_metrics.to_csv(RESULT_DIR / f"mixed_domain_{PRIMARY_MODEL}_{PRIMARY_FEATURE_SET}_metrics.csv", index=False)
combo_preds.to_csv(RESULT_DIR / f"mixed_domain_{PRIMARY_MODEL}_{PRIMARY_FEATURE_SET}_predictions.csv", index=False)

source_diversity_summary = plot_source_diversity(
    combo_metrics,
    path=FIG_DIR / f"source_diversity_{PRIMARY_MODEL}_{PRIMARY_FEATURE_SET}.png",
    feature_set=PRIMARY_FEATURE_SET,
    model=PRIMARY_MODEL,
)
source_diversity_summary.to_csv(RESULT_DIR / f"source_diversity_summary_{PRIMARY_MODEL}_{PRIMARY_FEATURE_SET}.csv", index=False)
display(source_diversity_summary)

,n_source_boards,includes_B5_source,rmse_mean,rmse_std,mae_mean,r2_mean
0,2,False,6.376910,2.751082,5.142674,0.942045
1,2,True,7.014416,3.011680,5.513229,0.930283
2,3,False,5.291151,2.444896,4.008281,0.959725
3,3,True,6.920670,2.755944,5.426927,0.933506
4,4,False,3.520431,NaN,2.372500,0.984978
5,4,True,6.651142,2.159151,5.173438,0.942140


In [12]:
# Direct robustness comparison: targets excluding B5, clean-only vs B5-inclusive sources.
robustness = combo_metrics.copy()
robustness_non_b5_targets = robustness[robustness["target_board"] != "B5"].copy()
robustness_summary = robustness_non_b5_targets.groupby(["n_source_boards", "includes_B5_source"], as_index=False).agg(
    rmse_mean=("rmse", "mean"),
    rmse_median=("rmse", "median"),
    mae_mean=("mae", "mean"),
    r2_mean=("r2", "mean"),
    n=("rmse", "size"),
)
robustness_summary.to_csv(RESULT_DIR / f"clean_vs_b5_inclusive_sources_{PRIMARY_MODEL}_{PRIMARY_FEATURE_SET}.csv", index=False)
display(robustness_summary)

,n_source_boards,includes_B5_source,rmse_mean,rmse_median,mae_mean,r2_mean,n
0,2,False,7.487902,7.313987,6.024531,0.924118,12
1,2,True,7.014416,6.581409,5.513229,0.930283,12
2,3,False,6.925588,6.092540,5.266563,0.935945,4
3,3,True,6.920670,5.634446,5.426927,0.933506,12
4,4,True,6.651142,6.002534,5.173438,0.942140,4


**Interpretation checkpoint.** If B5-inclusive training improves average non-B5 target performance, then the drifted/dynamically shifted board is not merely a liability. It expands the source manifold and can improve domain generalization.

## 6. Representation and model comparison

This section compares raw, physics-informed, and combined features across lightweight models. The expected Day2+ pattern is that representation quality dominates model complexity.

In [13]:
comparison = pd.concat([
    single_metrics.assign(experiment="single_source"),
    loo_metrics.assign(experiment="leave_one_out"),
], ignore_index=True)

summary = comparison.groupby(["experiment", "feature_set", "model"], as_index=False).agg(
    rmse_mean=("rmse", "mean"),
    rmse_median=("rmse", "median"),
    mae_mean=("mae", "mean"),
    r2_mean=("r2", "mean"),
)
summary.to_csv(RESULT_DIR / "representation_model_comparison.csv", index=False)
display(summary.sort_values(["experiment", "rmse_mean"]))

,experiment,feature_set,model,rmse_mean,rmse_median,mae_mean,r2_mean
1,leave_one_out,combined,RandomForest,5.733894,5.024069,4.366875,0.956143
2,leave_one_out,combined,XGBoost,6.007146,5.378613,4.480233,0.952514
4,leave_one_out,physics,RandomForest,6.025000,5.633029,4.613250,0.950708
5,leave_one_out,physics,XGBoost,6.092960,5.756510,4.574198,0.951047
8,leave_one_out,raw,XGBoost,6.887242,6.158908,5.232008,0.940152
7,leave_one_out,raw,RandomForest,7.604155,7.105163,5.895375,0.927781
6,leave_one_out,raw,LinearRegression,71.966204,63.391340,68.853457,-6.782209
3,leave_one_out,physics,LinearRegression,76.301022,34.029073,60.950545,-14.691247
0,leave_one_out,combined,LinearRegression,155.610579,110.724051,140.882195,-51.867621
13,single_source,physics,RandomForest,8.338336,7.777914,6.707094,0.903520


**Interpretation checkpoint.** If physics-informed features consistently outperform raw features across models, then the central Day2 conclusion remains intact: transfer is representation-limited more than model-limited.

## 7. PCA source-target geometry

These PCA plots visualize the feature manifold under source-target roles. They help distinguish simple baseline displacement from broader domain deformation.

In [14]:
# Day2 original stress case and one drift-inclusive reverse case.
pca_cases = [
    (["B1", "B2", "B3"], "B5", "day2_original_B123_to_B5"),
    (["B2", "B3", "B5"], "B1", "drift_inclusive_B235_to_B1"),
]

for sources, target, name in pca_cases:
    pca_df = plot_pca_by_board_role(
        df,
        source_boards=sources,
        target_board=target,
        feature_set=PRIMARY_FEATURE_SET,
        path=FIG_DIR / f"pca_{name}_{PRIMARY_FEATURE_SET}.png",
    )
    pca_df.to_csv(RESULT_DIR / f"pca_{name}_{PRIMARY_FEATURE_SET}.csv", index=False)

**Interpretation checkpoint.** If the target board lies outside the source cloud, extrapolation is required. If B5 broadens the source cloud toward other targets, it can improve robustness even if it is itself difficult to predict as a target.

## 8. Concentration-dependent transfer failures

This section checks whether transfer failures concentrate at high methane levels, where nonlinear saturation and recovery kinetics are most likely to matter.

In [15]:
failure_summary = plot_concentration_failure(
    loo_preds,
    feature_set=PRIMARY_FEATURE_SET,
    model=PRIMARY_MODEL,
    path=FIG_DIR / f"concentration_failure_loo_{PRIMARY_MODEL}_{PRIMARY_FEATURE_SET}.png",
)
failure_summary.to_csv(RESULT_DIR / f"concentration_failure_loo_{PRIMARY_MODEL}_{PRIMARY_FEATURE_SET}.csv", index=False)
display(failure_summary.head(20))

,source_boards,target_board,concentration_numeric,abs_error
0,B1+B2+B3+B4,B5,10,0.2000
1,B1+B2+B3+B4,B5,20,0.5375
2,B1+B2+B3+B4,B5,30,0.8875
3,B1+B2+B3+B4,B5,40,1.8625
4,B1+B2+B3+B4,B5,50,1.4750
5,B1+B2+B3+B4,B5,60,0.6375
6,B1+B2+B3+B4,B5,70,0.7125
7,B1+B2+B3+B4,B5,80,4.1125
8,B1+B2+B3+B4,B5,90,5.2625
9,B1+B2+B3+B4,B5,100,8.0375


**Interpretation checkpoint.** Increasing error at higher concentration suggests nonlinear saturation and/or recovery kinetics, rather than only baseline mismatch. Board-specific recovery residual features should be inspected when high-concentration failures dominate.

## 9. Focused scientific questions

The following cells produce compact tables for the Day2+ questions.

In [16]:
# Q1: easiest / hardest target boards under broad source training.
target_difficulty = loo_metrics[(loo_metrics["feature_set"] == PRIMARY_FEATURE_SET) & (loo_metrics["model"] == PRIMARY_MODEL)].copy()
target_difficulty = target_difficulty.sort_values("rmse")[["target_board", "source_boards", "rmse", "mae", "r2"]]
target_difficulty.to_csv(RESULT_DIR / "target_board_difficulty.csv", index=False)
display(target_difficulty)

# Q2: best single source boards averaged across targets.
source_quality = single_metrics[(single_metrics["feature_set"] == PRIMARY_FEATURE_SET) & (single_metrics["model"] == PRIMARY_MODEL)].copy()
source_quality = source_quality.groupby("source_boards", as_index=False).agg(
    rmse_mean=("rmse", "mean"), mae_mean=("mae", "mean"), r2_mean=("r2", "mean")
).sort_values("rmse_mean")
source_quality.to_csv(RESULT_DIR / "single_source_board_quality.csv", index=False)
display(source_quality)

# Q3/Q4: B5 as target vs B5 as source.
b5_target = loo_metrics[(loo_metrics["target_board"] == "B5") & (loo_metrics["feature_set"] == PRIMARY_FEATURE_SET) & (loo_metrics["model"] == PRIMARY_MODEL)]
b5_source = combo_metrics[(combo_metrics["includes_B5_source"]) & (combo_metrics["target_board"] != "B5")]
non_b5_source = combo_metrics[(~combo_metrics["includes_B5_source"]) & (combo_metrics["target_board"] != "B5")]

b5_role_summary = pd.DataFrame({
    "question": ["B5 as held-out target RMSE", "B5-inclusive source mean RMSE to non-B5", "clean-only source mean RMSE to non-B5"],
    "value": [b5_target["rmse"].mean(), b5_source["rmse"].mean(), non_b5_source["rmse"].mean()],
})
b5_role_summary.to_csv(RESULT_DIR / "b5_role_summary.csv", index=False)
display(b5_role_summary)

,target_board,source_boards,rmse,mae,r2
24,B5,B1+B2+B3+B4,3.520431,2.372500,0.984978
23,B4,B1+B2+B3+B5,4.847203,4.092500,0.971521
22,B3,B1+B2+B4+B5,5.633029,4.550625,0.961538
21,B2,B1+B3+B4+B5,6.372040,5.101875,0.950784
20,B1,B2+B3+B4+B5,9.752295,6.948750,0.884718


,source_boards,rmse_mean,mae_mean,r2_mean
1,B2,6.609944,5.624219,0.945220
2,B3,7.068957,5.289062,0.925847
4,B5,7.944368,6.139687,0.912607
0,B1,8.392994,7.476562,0.911720
3,B4,11.675417,9.005937,0.822205


,question,value
0,B5 as held-out target RMSE,3.520431
1,B5-inclusive source mean RMSE to non-B5,6.922343
2,clean-only source mean RMSE to non-B5,7.347324


## 10. Day2+ conclusions template

Edit the statements below after inspecting the generated tables and figures.

1. **Transfer asymmetry:** Single-source board-to-board transfer should be interpreted directionally, not as a symmetric distance.
2. **B5 scientific role:** B5 should not be framed as a bad sensor. It is a drifted/dynamically shifted board that may be difficult as a target but informative as a source.
3. **Source diversity:** Broader source-domain manifolds are expected to improve generalization when the target lies within or near the expanded manifold.
4. **Representation:** Physics-informed features remain the primary source of transfer improvement; nonlinear models help but do not remove domain mismatch.
5. **Failure mechanism:** Concentration-dependent errors and PCA deformation should be used together to distinguish baseline mismatch, dynamic mismatch, nonlinear saturation, and recovery-kinetic distortion.

In [17]:
# Final path and artifact check.
expected_files = [
    RESULT_DIR / "feature_leakage_check.json",
    RESULT_DIR / "single_source_transfer_metrics.csv",
    RESULT_DIR / "leave_one_board_out_metrics.csv",
    RESULT_DIR / f"mixed_domain_{PRIMARY_MODEL}_{PRIMARY_FEATURE_SET}_metrics.csv",
    FIG_DIR / f"single_source_{PRIMARY_MODEL}_{PRIMARY_FEATURE_SET}_rmse_heatmap.png",
    FIG_DIR / f"asymmetry_{PRIMARY_MODEL}_{PRIMARY_FEATURE_SET}.png",
    FIG_DIR / f"source_diversity_{PRIMARY_MODEL}_{PRIMARY_FEATURE_SET}.png",
]

for f in expected_files:
    print(f.exists(), f.relative_to(PROJECT_ROOT))

print("Day2+ notebook completed. Review the interpretation checkpoints and generated CSV/PNG outputs.")

True results\day2plus\feature_leakage_check.json
True results\day2plus\single_source_transfer_metrics.csv
True results\day2plus\leave_one_board_out_metrics.csv
True results\day2plus\mixed_domain_RandomForest_physics_metrics.csv
True figures\day2plus\single_source_RandomForest_physics_rmse_heatmap.png
True figures\day2plus\asymmetry_RandomForest_physics.png
True figures\day2plus\source_diversity_RandomForest_physics.png
Day2+ notebook completed. Review the interpretation checkpoints and generated CSV/PNG outputs.
